In [ ]:
# raster-genie — shared config (%run ./config_nb). Sentinel-2 + WorldPop master COGs
# landed as FILE columns in geospatial_catalog.raster_genie, queried with the gbx_rst_*
# functions in SQL, then asked about through a Genie Agent. Sets catalog/schema + the Volume
# tree + toggles + downloaders. Mirrors the vapor-eyes config_nb (comments + minimal
# prints, no markdown cells).

# -- GeoBrix: two-step install so a freshly
#    rebuilt same-version wheel's bytes replace the cached install (step 1
#    --force-reinstall --no-deps) and the extras still resolve (step 2). Then a
#    %restart_python (next cell) loads the fresh bytes.
%pip install --quiet --disable-pip-version-check --force-reinstall --no-deps "geobrix @ file:///Volumes/geospatial_catalog/geobrix/sample-data/env6/geobrix-0.6.0+env6-py3-none-any.whl"
%pip install --quiet "geobrix[stac,vizx,databricks] @ file:///Volumes/geospatial_catalog/geobrix/sample-data/env6/geobrix-0.6.0+env6-py3-none-any.whl"

In [ ]:
%restart_python

In [ ]:
import os

from pyspark.sql import functions as F
from pyspark.sql.types import *

# GeoBrix: one Python API and one register call on every compute.
from databricks.labs import gbx
from databricks.labs.gbx.rasterx import functions as rx  # rx.rst_* (raster)

_ = gbx.register(spark)  # the gbx_* SQL functions and the *_gbx readers and writers

In [ ]:
# ============================================================
# USER SETTINGS — edit these (everything below is wired off them)
# ============================================================

# Unity Catalog: the schema holds the tables; its Volume 'data' holds the COGs (created below).
catalog_name = "geospatial_catalog"
schema_name = "raster_genie"

# Toggle (overridable per-notebook right after %run ./config_nb):
FORCE_REBUILD = False       # True -> re-download / re-create tables (skip-guards off)

# The event: the 2024 Smokehouse Creek fire in the Texas Panhandle (ignited 2024-02-26).
FIRE_NAME = "SMOKEHOUSE CREEK"
FIRE_STATE = "TX"
FIRE_YEAR = 2024
FIRE_AOI_ID = "smokehouse_creek"

# Sentinel-2 windows: a scene before the fire and a scene after it (STAC datetime ranges).
PRE_WINDOW = "2024-02-01/2024-02-25"
POST_WINDOW = "2024-03-05/2024-03-31"
MAX_CLOUD = 20              # keep scenes below this % scene-level cloud cover
BBOX_PAD = 0.1              # degrees of margin around the perimeter bbox for the downloads

# Burn-severity read size. gbx_rst_fromfile picks the coarsest overview at least this fine
# (a 10 m COG has 20/40/80/160 m overviews). 40 m keeps one scene's window under the 64 MiB
# a Unity Catalog function can return; the map uses the coarser MAP_RES_M.
BURN_RES_M = 40.0
MAP_RES_M = 80.0

# The Genie Agent (Agent mode). Paste the space id of the raster-genie Genie space here;
# notebook 01 skips its last step while this is empty.
GENIE_SPACE_ID = ""

In [ ]:
# Apply catalog/schema from USER SETTINGS; the Volume holds the master COGs.
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema_name}")
spark.sql(f"USE DATABASE {schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.data")
print(f"... catalog: '{catalog_name}' (USE)")
print(f"... schema:  '{schema_name}' (CREATE / USE)")
print(f"... volume:  '{catalog_name}.{schema_name}.data' (CREATE IF NOT EXISTS)")

In [ ]:
# Volume ETL tree. Bare /Volumes paths only.
ETL_DIR = f"/Volumes/{catalog_name}/{schema_name}/data"
RASTER_GENIE_DIR = f"{ETL_DIR}/raster-genie"
CONTEXT_DIR = f"{RASTER_GENIE_DIR}/context"         # TIGER counties + state
FIRE_DIR = f"{RASTER_GENIE_DIR}/fire"               # the perimeter GeoJSON
S2_RAW_DIR = f"{RASTER_GENIE_DIR}/s2_bands"         # windowed per-band downloads
S2_MASTER_DIR = f"{RASTER_GENIE_DIR}/s2_master"     # one multi-band master COG per scene
POP_RAW_DIR = f"{RASTER_GENIE_DIR}/population_raw"  # the windowed WorldPop GeoTIFF
POP_CACHE_DIR = f"{RASTER_GENIE_DIR}/population_src"  # the national WorldPop file, fetched once
POP_DIR = f"{RASTER_GENIE_DIR}/population"          # the population master COG
for _d in (CONTEXT_DIR, FIRE_DIR, S2_RAW_DIR, S2_MASTER_DIR, POP_RAW_DIR, POP_CACHE_DIR, POP_DIR):
    dbutils.fs.mkdirs(_d)
print(f"... RASTER_GENIE_DIR: '{RASTER_GENIE_DIR}' (MKDIRS context/ fire/ s2_bands/ s2_master/ population*/)")
print(f"... fire: {FIRE_NAME} ({FIRE_STATE}, {FIRE_YEAR})  pre: {PRE_WINDOW}  post: {POST_WINDOW}  MAX_CLOUD: {MAX_CLOUD}")
print(f"... FORCE_REBUILD={FORCE_REBUILD}  BURN_RES_M={BURN_RES_M}  MAP_RES_M={MAP_RES_M}")

In [ ]:
# Idempotent managed-Delta materializer (Serverless-safe cache() stand-in).
def finalize_delta(df, tbl_name, do_display=True):
    if FORCE_REBUILD:
        spark.sql(f"DROP TABLE IF EXISTS {tbl_name}")
    elif spark.catalog.tableExists(tbl_name):
        if [f.name for f in spark.table(tbl_name).schema] != [
            f.name for f in df.schema
        ]:
            print(f"... schema changed for {tbl_name} -> rewriting (was stale)")
            spark.sql(f"DROP TABLE IF EXISTS {tbl_name}")
    if not spark.catalog.tableExists(tbl_name):
        df.write.mode("overwrite").saveAsTable(tbl_name)
        print(f"... wrote table {tbl_name} ({spark.table(tbl_name).count():,} rows)")
    else:
        print(f"... table {tbl_name} exists (skip; FORCE_REBUILD=False)")
    out = spark.table(tbl_name)
    if do_display:
        out.printSchema()
    return out

In [ ]:
# Downloaders + STAC client + FILE writer + Genie Agent client + viz helpers.
from databricks.labs.gbx.ds.file_gbx import gbx_file_write
from databricks.labs.gbx.genie import ask as ask_genie
from databricks.labs.gbx.rasterx.helpers import prepare_cogs
from databricks.labs.gbx.sample import FirePerimeterDownloader, WorldPopDownloader
from databricks.labs.gbx.stac import StacClient
from databricks.labs.gbx.vizx import plot_tile

fire_perimeters = FirePerimeterDownloader()
worldpop = WorldPopDownloader()
stac_client = StacClient()

print(
    "... downloaders: fire_perimeters, worldpop | stac_client | gbx_file_write | "
    "prepare_cogs | ask_genie | vizx: plot_tile"
)